# Time Series Caching Verification

In [2]:
import sys, os, json
from datetime import date
sys.path.insert(0, os.path.abspath('../..'))

from apps.agentic.core.utils import set_anthropic_env
from apps.agentic.core.series_cache import SeriesCache

set_anthropic_env(filedir='../../.keys')

## SeriesCache

In [3]:
# Initialize
SeriesCache.initialize()

# Put a test entry
fake_data = {
    "observations": [
        {"date": "2020-01-01", "value": "100.0"},
        {"date": "2020-02-01", "value": "101.5"},
    ]
}

cache_id = await SeriesCache.put(
    source="fred",
    native_id="TEST_SERIES",
    data=fake_data,
    observation_start="2020-01-01",
    observation_end="2020-02-01",
    observation_count=2,
    frequency="Monthly",
    units="Index",
    title="Test Series",
)
print(f"cache_id: {cache_id}")

INFO:     SeriesCache initialized: postgresql://yada@localhost/yada
DEBUG:    SeriesCache: stored fred:TEST_SERIES → 19d593ca-f355-4da6-a764-a3d8d7bf761a
cache_id: 19d593ca-f355-4da6-a764-a3d8d7bf761a


In [5]:
# Fetch by cache_id
by_id = await SeriesCache.get_by_cache_id(cache_id)
assert by_id is not None
print(f"get_by_cache_id: {by_id['native_id']} — {by_id['observation_count']} obs")

# Fetch by native_idassert by_id is not None
by_native = await SeriesCache.get_by_native_id("fred", "TEST_SERIES")
assert by_native is not None
print(f"get_by_native_id: {by_native['cache_id']} — expires {by_native['expires_at'].date()}")

# Confirm same cache_id
assert str(by_native['cache_id']) == cache_id
print("OK — both lookups return the same entry")

get_by_cache_id: TEST_SERIES — 2 obs
get_by_native_id: 19d593ca-f355-4da6-a764-a3d8d7bf761a — expires 2026-05-26
OK — both lookups return the same entry


## SeriesRef

In [6]:
from apps.agentic.core.series_ref import SeriesRef

ref = SeriesRef(source="fred", native_id="UNRATE", cache_id="19d593ca-f355-4da6-a764-a3d8d7bf761a")

json_str = ref.to_json()
print(f"to_json:   {json_str}")

ref2 = SeriesRef.from_json(json_str)
print(f"from_json: {ref2}")

assert ref == ref2
print("OK — round-trip preserved all fields")

to_json:   {"source": "fred", "native_id": "UNRATE", "cache_id": "19d593ca-f355-4da6-a764-a3d8d7bf761a"}
from_json: SeriesRef(source='fred', native_id='UNRATE', cache_id='19d593ca-f355-4da6-a764-a3d8d7bf761a')
OK — round-trip preserved all fields


## CachingFredTool

In [3]:
from apps.agentic.core.series_cache import SeriesCache
from apps.agentic.agents.data.caching_fred_tool import CachingFredTool
from lib.mcp_client import MCPClient, MCPClientConfig
from langchain_mcp_adapters.client import MultiServerMCPClient
from apps.agentic.core.constants import MCP_URL

SeriesCache.initialize()

# Get the real MCP tool
mcp_client = MultiServerMCPClient({"fred": {"transport": "sse", "url": MCP_URL}}) # type: ignore
mcp_tools = await mcp_client.get_tools()
fred_tool = next(t for t in mcp_tools if t.name == "fred_series_observations")

tool = CachingFredTool(wrapped=fred_tool)

INFO:     SeriesCache initialized: postgresql://yada@localhost/yada


In [4]:
# Cache miss — should fetch from MCP and store
result1 = await tool._arun(series_id="APU0000708111")
print(f"Miss result: {result1}")

# Cache hit — should return immediately without MCP call
result2 = await tool._arun(series_id="APU0000708111")
print(f"Hit result:  {result2}")
assert result1 == result2
print("OK — same SeriesRef returned on both calls")

DEBUG:    fred_series_observations: cache miss for fred:APU0000708111 — fetching
DEBUG:    SeriesCache: stored fred:APU0000708111 → c8fdf439-2d8d-4fb6-8171-e78a218c7afd
DEBUG:    fred_series_observations: cached fred:APU0000708111 → c8fdf439-2d8d-4fb6-8171-e78a218c7afd
Miss result: {"source": "fred", "native_id": "APU0000708111", "cache_id": "c8fdf439-2d8d-4fb6-8171-e78a218c7afd"}
DEBUG:    fred_series_observations: cache hit for fred:APU0000708111
Hit result:  {"source": "fred", "native_id": "APU0000708111", "cache_id": "c8fdf439-2d8d-4fb6-8171-e78a218c7afd"}
OK — same SeriesRef returned on both calls


# DataFetcherAgent

In [5]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
import shortuuid

from apps.agentic.core.mcp_tool_registry import MCPToolRegistry
from apps.agentic.core.constants import MCP_URL
from apps.agentic.agents.data.data_fetcher_agent import DataFetcherAgent

await MCPToolRegistry.initialize(
    servers={"fred": {"transport": "sse", "url": MCP_URL}},
    required=["fred_series_observations"],
)

agent = await DataFetcherAgent.create()
print(f"Tools: {list(agent.tool_node.tools_by_name.keys())}")

DEBUG:    MCPToolRegistry discovered 6 tools: ['fred_category_children', 'fred_category_series', 'fred_series_info', 'fred_series_observations', 'fred_series_updates', 'fred_release_series']
INFO:     MCPToolRegistry initialized with 6 tools: ['fred_category_children', 'fred_category_series', 'fred_series_info', 'fred_series_observations', 'fred_series_updates', 'fred_release_series']
DEBUG:    DataFetcherAgent using MCP tools: ['fred_series_observations']
DEBUG:    DataFetcherAgent prompt: 
            <instructions>
            You are a data fetcher agent. Use the available MCP tools to retrieve time series
            data from external data sources.

            The tools return a SeriesRef JSON object — pass it back unmodified as your response.
            Do not summarize, reformat, or interpret it.

            When fetching FRED data use the fred_series_observations tool with a limit of
            10000 to retrieve the full series.
            </instructions>
            
Too

In [6]:
state = {"messages": [HumanMessage(content="Fetch the UNRATE series from FRED.")]}
config = RunnableConfig(configurable={"thread_id": shortuuid.uuid()})

result = await agent.agent.ainvoke(state, config)
last = result["messages"][-1]
print(f"Last message type: {type(last).__name__}")
print(f"Content: {last.content}")


INFO:     [DataFetcherAgent] routing → fred_series_observations({'series_id': 'UNRATE', 'limit': 10000})
DEBUG:    fred_series_observations: cache miss for fred:UNRATE — fetching
DEBUG:    SeriesCache: stored fred:UNRATE → 3ed4a7c3-4f0e-4f2c-8fd4-091c31275ef2
DEBUG:    fred_series_observations: cached fred:UNRATE → 3ed4a7c3-4f0e-4f2c-8fd4-091c31275ef2
Last message type: ToolMessage
Content: {"source": "fred", "native_id": "UNRATE", "cache_id": "3ed4a7c3-4f0e-4f2c-8fd4-091c31275ef2"}


## TimeSeriesPlotAgent

In [7]:
from apps.agentic.core.series_cache import SeriesCache
from apps.agentic.core.series_ref import SeriesRef
from apps.agentic.agents.plots.time_series_plot_agent import _load_series

SeriesCache.initialize()

# Use the BOGMBBM entry already in the cache from step 3 verification
entry = await SeriesCache.get_by_native_id("fred", "BOGMBBM")
assert entry is not None
ref = SeriesRef(source="fred", native_id="BOGMBBM", cache_id=str(entry["cache_id"]))

# Full series
times, values = _load_series(ref.to_json(), date_from=None, date_to=None)
print(f"Full series: {len(times)} observations, {times[0].date()} → {times[-1].date()}")

# Date filtered
times_f, values_f = _load_series(ref.to_json(), date_from="2000-01-01", date_to="2010-12-31")
print(f"Filtered:    {len(times_f)} observations, {times_f[0].date()} → {times_f[-1].date()}")

INFO:     SeriesCache initialized: postgresql://yada@localhost/yada
Full series: 806 observations, 1959-01-01 → 2026-02-01
Filtered:    132 observations, 2000-01-01 → 2010-12-01
